(ch:ml-wine)=
# 분류: 와인 품질 데이터셋

이 장에서는 UCI Machine Learning Repository의 레드 와인 품질 데이터셋을 이용하여 이진 분류 모델을 만든다. 목표는 품질 점수가 높은 와인을 찾아내는 모델을 훈련하는 것뿐 아니라, 정확도, 혼동 행렬, 정밀도, 재현율을 함께 해석하는 것이다.

## 데이터셋

와인 품질 데이터셋은 포르투갈 비뉴 베르드 와인의 물리화학적 측정값과 품질 점수를 담고 있다.

- 입력 특성: 산도, 당도, 염화물, 이산화황, 밀도, pH, 황산염, 알코올 등 11개 수치형 특성
- 원래 타깃: `quality`, 0에서 10 사이의 품질 점수
- 이진 타깃: `quality >= 7`이면 `good`, 그렇지 않으면 `ordinary`

이 데이터셋은 좋은 와인이 상대적으로 적기 때문에, 정확도만으로 모델을 평가하면 중요한 부분을 놓칠 수 있다.

필요한 라이브러리와 레드 와인 품질 데이터셋을 불러온다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_theme(style="whitegrid")

UCI에서 제공하는 CSV 파일을 읽고, 품질 점수를 기준으로 이진 타깃을 만든다.

In [ ]:
wine_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_df = pd.read_csv(wine_url, sep=";")

wine_df["quality_label"] = (wine_df["quality"] >= 7).map({True: "good", False: "ordinary"})

X = wine_df.drop(columns=["quality", "quality_label"])
y_quality = wine_df["quality_label"]

wine_df.head()

데이터의 크기와 타깃 클래스의 개수를 확인한다.

In [ ]:
wine_df.shape

In [ ]:
wine_df["quality_label"].value_counts()

입력 특성은 모두 수치형이다. 여기서는 지표 해석이 잘 보이도록 전체 11개 특성 중 일부만 선택해서 먼저 모델을 만든다.

In [ ]:
X.dtypes.value_counts()

## 주요 특성 선택

전체 특성을 모두 사용할 수도 있지만, 여기서는 설명을 위해 와인 품질과 관련이 있어 보이는 네 개의 특성만 먼저 사용한다.

In [ ]:
selected_features = [
    "alcohol",
    "volatile acidity",
    "sulphates",
    "density",
]

wine_df[selected_features + ["quality", "quality_label"]].head()

선택한 특성의 클래스별 분포를 상자그림으로 비교한다. 어떤 특성은 좋은 와인과 보통 이하 와인의 분포가 어느 정도 다르고, 어떤 특성은 서로 겹치는 부분이 많다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for feature, ax in zip(selected_features, axes.ravel()):
    sns.boxplot(data=wine_df, x="quality_label", y=feature, ax=ax)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

알코올 도수와 휘발성 산도를 산점도로 그려 두 클래스가 어느 정도 구분되는지 확인한다.

In [ ]:
sns.scatterplot(
    data=wine_df,
    x="alcohol",
    y="volatile acidity",
    hue="quality_label",
    alpha=0.7,
)
plt.title("Wine quality by alcohol and volatile acidity")
plt.show()

## 모델 훈련

선택한 특성만 사용하여 훈련셋과 테스트셋을 나눈다. 좋은 와인의 비율이 적으므로 `stratify`를 사용하여 두 셋의 클래스 비율이 비슷하게 유지되도록 한다.

In [ ]:
X_selected = wine_df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y_quality,
    test_size=0.3,
    random_state=42,
    stratify=y_quality,
)

특성마다 값의 범위가 다르므로 표준화를 적용한다. 표준화 기준은 훈련셋에서 계산하고 테스트셋에는 같은 기준을 적용한다.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
wine_model = LogisticRegression(max_iter=1000)
wine_model.fit(X_train_scaled, y_train)

테스트셋에 대해 예측하고 정확도를 계산한다. 정확도는 전체 테스트 샘플 중 맞게 예측한 비율이다.

In [ ]:
y_pred = wine_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

accuracy

## 혼동 행렬

혼동 행렬은 모델이 어떤 클래스를 어떤 클래스로 예측했는지 보여준다. 좋은 와인을 놓쳤는지, 보통 이하 와인을 좋은 와인이라고 잘못 골랐는지 확인할 수 있다.

In [ ]:
labels = wine_model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## 정밀도와 재현율

정밀도는 모델이 어떤 클래스로 예측한 것 중 실제로 맞은 비율이다. 재현율은 실제 그 클래스인 샘플 중 모델이 찾아낸 비율이다.

좋은 와인이 적은 데이터에서는 모델이 대부분을 `ordinary`로 예측해도 정확도가 높아 보일 수 있다. 따라서 정확도 하나만 보기보다 정밀도와 재현율을 함께 확인해야 한다.

In [ ]:
print(classification_report(y_test, y_pred))

## 전체 특성과 비교

이번에는 전체 11개 특성을 사용하여 같은 모델을 훈련한다. 주요 특성 몇 개만 사용했을 때와 전체 특성을 사용했을 때 성능이 어떻게 달라지는지 비교한다.

In [ ]:
X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(
    X,
    y_quality,
    test_size=0.3,
    random_state=42,
    stratify=y_quality,
)

all_scaler = StandardScaler()
X_all_train_scaled = all_scaler.fit_transform(X_all_train)
X_all_test_scaled = all_scaler.transform(X_all_test)

wine_all_model = LogisticRegression(max_iter=1000)
wine_all_model.fit(X_all_train_scaled, y_all_train)

y_all_pred = wine_all_model.predict(X_all_test_scaled)
all_accuracy = accuracy_score(y_all_test, y_all_pred)

pd.DataFrame(
    {
        "model": ["selected features", "all features"],
        "accuracy": [accuracy, all_accuracy],
    }
)

## 정리

이번 장에서는 와인 품질 데이터셋으로 이진 분류 모델과 평가 지표를 살펴보았다.

- 이진 분류는 두 클래스 중 하나를 예측하는 문제이다.
- 클래스 비율이 치우친 데이터에서는 정확도만으로 모델을 평가하기 어렵다.
- 혼동 행렬은 모델이 어떤 종류의 실수를 하는지 보여준다.
- 정밀도는 예측한 것 중 얼마나 맞았는지를 나타낸다.
- 재현율은 실제 해당 클래스 중 얼마나 찾아냈는지를 나타낸다.
- 문제의 맥락에 따라 정밀도와 재현율 중 더 중요하게 볼 지표가 달라질 수 있다.

## 연습문제

**문제 1**

`alcohol`, `sulphates` 두 특성만 사용하여 모델을 훈련해 보아라. 네 개 특성을 사용했을 때와 성능이 어떻게 달라지는가?

**문제 2**

혼동 행렬에서 모델이 좋은 와인을 얼마나 놓치는지 설명해 보아라.

**문제 3**

`classification_report()`에서 `good`과 `ordinary`의 정밀도와 재현율을 비교해 보아라. 두 클래스의 지표가 같은가, 다른가?